# Lab 20 — Introducción a Qiskit Pulse y Control de Hardware

Qiskit Pulse permite controlar qubits a nivel de microondas: definir la forma del pulso,
la frecuencia de la portadora y el calendario de envío.

**Nota**: La ejecución en hardware real requiere acceso a IBM Quantum. Este lab muestra cómo construir y analizar los schedules sin conexión al hardware.

## 1. Conceptos clave de Qiskit Pulse

| Concepto | Descripción |
|----------|-------------|
| **Drive channel** | Canal que envía pulsos al qubit |
| **Measure channel** | Canal para la señal de medición |
| **Acquire channel** | Canal para capturar la respuesta |
| **Waveform** | Forma del pulso en el tiempo |
| **Schedule** | Calendario de pulsos con tiempos absolutos |
| **ScheduleBlock** | Calendario con tiempos relativos (más moderno) |

La frecuencia característica del qubit es ω₀ ≈ 4-6 GHz. El pulso π tiene duración típica de 50-200 ns en hardware real.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from qiskit import pulse
from qiskit.pulse import (
    Schedule, ScheduleBlock, DriveChannel, MeasureChannel, AcquireChannel,
    Gaussian, GaussianSquare, Drag, Constant
)

# Formas de pulso comunes en hardware
dt = 2/9e8  # ~ 2.22 ns (tiempo de muestra IBM)
duration = 160  # muestras (~ 356 ns)

# Gaussiano: estándar para rotaciones de 1 qubit
gauss = Gaussian(duration=duration, amp=0.8, sigma=40)

# Gaussiano-cuadrado: usado para puertas de 2 qubits (CR)
gauss_sq = GaussianSquare(duration=duration*2, amp=0.3, sigma=40, risefall_sigma_ratio=2)

# DRAG: corrige leakage al nivel |2⟩ (derivada gaussiana)
drag = Drag(duration=duration, amp=0.8, sigma=40, beta=10)

print(f'Duración gaussiano: {duration} muestras = {duration * dt * 1e9:.1f} ns')
print(f'Duración GaussSquare: {duration*2} muestras = {duration*2 * dt * 1e9:.1f} ns')
print(f'dt = {dt*1e9:.3f} ns (período de muestreo)')

## 2. Visualización de formas de pulso

Graficamos la amplitud en fase (I) y cuadratura (Q) de cada forma.
La envolvente compleja A(t) = I(t) + iQ(t) modula la portadora RF.

In [ ]:
def get_samples(pulse_obj, n_samples=None):
    """Extrae muestras complejas de un objeto pulse."""
    dur = pulse_obj.duration
    if n_samples:
        dur = min(dur, n_samples)
    return pulse_obj.get_waveform().samples[:dur]

pulses = {
    'Gaussiano': gauss,
    'GaussSquare': gauss_sq,
    'DRAG': drag,
}

fig, axes = plt.subplots(2, 3, figsize=(14, 7))
for col, (name, p) in enumerate(pulses.items()):
    samples = get_samples(p)
    times = np.arange(len(samples)) * dt * 1e9

    axes[0, col].plot(times, samples.real, 'b-', label='I (fase)')
    axes[0, col].plot(times, samples.imag, 'r--', label='Q (cuadratura)', alpha=0.7)
    axes[0, col].set_title(name, fontsize=12)
    axes[0, col].legend(fontsize=9)
    axes[0, col].set_xlabel('Tiempo (ns)')
    axes[0, col].set_ylabel('Amplitud')
    axes[0, col].grid(alpha=0.3)

    # Espectro de frecuencias
    spectrum = np.abs(np.fft.fftshift(np.fft.fft(samples)))
    freqs = np.fft.fftshift(np.fft.fftfreq(len(samples), d=dt)) / 1e6  # MHz
    axes[1, col].plot(freqs, spectrum / spectrum.max(), 'g-')
    axes[1, col].set_title(f'Espectro {name}', fontsize=12)
    axes[1, col].set_xlabel('Frecuencia (MHz)')
    axes[1, col].set_ylabel('Amplitud norm.')
    axes[1, col].set_xlim(-50, 50)
    axes[1, col].grid(alpha=0.3)

plt.suptitle('Formas de pulso y sus espectros', fontsize=14)
plt.tight_layout()
plt.show()

## 3. Construcción de un ScheduleBlock

Un `ScheduleBlock` define la secuencia de pulsos en el tiempo.
En Qiskit Pulse moderno, las operaciones se añaden usando el context manager `with pulse.build()`.

In [ ]:
# Construir un schedule para un π-pulso y medición
with pulse.build(name='pi_pulse_and_measure') as pi_schedule:
    d0 = pulse.DriveChannel(0)
    m0 = pulse.MeasureChannel(0)
    a0 = pulse.AcquireChannel(0)

    # Aplicar π-pulso gaussiano en qubit 0
    pulse.play(gauss, d0)

    # Añadir delay para que el estado se estabilice
    pulse.delay(40, d0)

    # Medición
    meas_gauss = GaussianSquare(duration=800, amp=0.2, sigma=64,
                               risefall_sigma_ratio=2)
    pulse.play(meas_gauss, m0)
    pulse.acquire(800, a0, pulse.MemorySlot(0))

print(pi_schedule)
print(f'\nDuración total del schedule: {pi_schedule.duration} muestras')
print(f'  = {pi_schedule.duration * dt * 1e9:.1f} ns')

## 4. Secuencia Rabi con Qiskit Pulse

Para calibrar el π-pulso, creamos múltiples schedules con diferentes amplitudes
y medimos P(|1⟩) en función de la amplitud. Este es el protocolo exacto que
usan los experimentos de calibración en hardware IBM.

In [ ]:
amplitudes = np.linspace(0, 1.0, 20)
schedules = []

for amp in amplitudes:
    with pulse.build(name=f'rabi_amp_{amp:.2f}') as sched:
        d0 = pulse.DriveChannel(0)
        m0 = pulse.MeasureChannel(0)
        a0 = pulse.AcquireChannel(0)

        # Pulso de amplitud variable
        rabi_pulse = Gaussian(duration=160, amp=amp, sigma=40)
        pulse.play(rabi_pulse, d0)
        pulse.delay(40, d0)

        meas = GaussianSquare(duration=800, amp=0.2, sigma=64, risefall_sigma_ratio=2)
        pulse.play(meas, m0)
        pulse.acquire(800, a0, pulse.MemorySlot(0))

    schedules.append(sched)

print(f'Creados {len(schedules)} schedules para barrido Rabi')
print(f'Amplitudes: {amplitudes[0]:.2f} → {amplitudes[-1]:.2f}')

# Predicción teórica: theta ∝ amplitud, pi-pulse a amp = 0.5 (normalmente)
theta_equiv = amplitudes * np.pi  # amplitud 1.0 = 2π, 0.5 = π
p1_predicted = np.sin(theta_equiv / 2)**2

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(amplitudes, p1_predicted, 'bo-', markersize=6, linewidth=2)
ax.axvline(0.5, color='red', linestyle='--', label='π-pulso (amp=0.5)')
ax.set_xlabel('Amplitud del pulso', fontsize=12)
ax.set_ylabel('P(|1⟩) predicho', fontsize=12)
ax.set_title('Curva Rabi esperada con barrido de amplitud', fontsize=13)
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

## 5. Notas sobre ejecución en hardware real

Para ejecutar estos schedules en hardware IBM, se necesita:

```python
from qiskit_ibm_runtime import QiskitRuntimeService
service = QiskitRuntimeService(channel='ibm_quantum',
                               token=os.environ['IBM_QUANTUM_TOKEN'])
backend = service.least_busy(operational=True, simulator=False,
                             filters=lambda b: b.open_pulse)
```

No todos los backends ofrecen acceso Pulse (`open_pulse=True`).
Los backends con `open_pulse` suelen ser los más avanzados (Eagle, Heron).

In [ ]:
# Resumen de propiedades de pulso
print('Resumen del protocolo de calibración Rabi con Qiskit Pulse:')
print()
print('1. Crear schedules con amplitudes A ∈ [0, A_max]')
print('2. Ejecutar en hardware y medir P(|1⟩) para cada A')
print('3. Ajustar P = sin²(Ω·A/2) para extraer Ω')
print('4. π-pulso: A_pi = π/Ω')
print('5. Validar: aplicar A_pi → P(|1⟩) ≈ 1.0')
print()
print('Formas de pulso y sus usos en IBM:')
print('  Gaussian     → puertas de 1 qubit (X, Y, Z, H)')
print('  GaussSquare  → puertas CR (Cross-Resonance) para CNOT')
print('  DRAG         → corrige fuga al nivel |2⟩ del transmon')
print('  Constant     → pulsos de calibración de fase')